# Limpieza — Fase 2: Texto libre


In [1]:
import os
import sys
import re

import pandas as pd

sys.path.append(os.path.join("..", "scripts"))
from limpieza_utils import CARPETA_INTERIM, es_vacio, es_faltante, normalizar_espacios, marcar_faltantes_como_na, RegistroTransformaciones

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

RUTA_ENTRADA = os.path.join(CARPETA_INTERIM, "fase1_identificadores.csv")
df = pd.read_csv(RUTA_ENTRADA, dtype=str, keep_default_na=False, encoding="utf-8-sig")

log = RegistroTransformaciones("Fase 2 - Texto libre")

print(f"Filas: {len(df)}  |  Columnas: {df.shape[1]}")


Filas: 11867  |  Columnas: 18


## ESTABLECIMIENTO

Problemas: 1,392 filas con espacios dobles internos; 2,225 con comillas dobles y 747 con apóstrofe (mezcla de comillas decorativas alrededor de un alias y apóstrofes ortográficos legítimos de nombres k'iche').


In [2]:
antes_espacio_doble = df["ESTABLECIMIENTO"].str.contains(r"  ", regex=True, na=False).sum()

df["ESTABLECIMIENTO"] = normalizar_espacios(df["ESTABLECIMIENTO"])

assert not df["ESTABLECIMIENTO"].str.contains(r"  ", regex=True, na=False).any(), "Quedaron espacios dobles en ESTABLECIMIENTO"
assert not df["ESTABLECIMIENTO"].str.contains(r"^\s|\s$", regex=True, na=False).any(), "Quedaron espacios al borde en ESTABLECIMIENTO"

log.registrar(
    "ESTABLECIMIENTO", f"{antes_espacio_doble} filas con espacios dobles internos",
    "Trim + colapso de espacios múltiples a uno solo", antes_espacio_doble,
    "Limpieza de formato estándar; no cambia el contenido, solo el espaciado.",
)
print(f"ESTABLECIMIENTO: {antes_espacio_doble} filas con espacios dobles corregidas.")


[Fase 2 - Texto libre] ESTABLECIMIENTO: Trim + colapso de espacios múltiples a uno solo -> 1392 registros afectados
ESTABLECIMIENTO: 1392 filas con espacios dobles corregidas.


In [3]:
no_mayus = df["ESTABLECIMIENTO"].str.contains(r"[a-z]", regex=True, na=False).sum()
print(f"Filas con alguna minúscula: {no_mayus}")

log.registrar(
    "ESTABLECIMIENTO", "Mayúsculas: se verifica consistencia (no se fuerza un cambio de estilo innecesario)",
    f"Sin cambios de mayúsculas ({no_mayus} filas con minúscula sí encontradas; se preservan tal cual, podrían ser parte de un nombre propio)" if no_mayus else "Sin cambios: el campo ya está 100% en mayúsculas",
    no_mayus,
    "La fuente ya entrega el campo casi enteramente en mayúsculas; forzar mayúsculas sobre casos minoritarios sin revisar podría alterar un nombre propio con capitalización intencional.",
)


Filas con alguna minúscula: 0
[Fase 2 - Texto libre] ESTABLECIMIENTO: Sin cambios: el campo ya está 100% en mayúsculas -> 0 registros afectados


In [4]:
LIMITE_ANTES = r"(^|(?<=[\s.,;:()\-]))"
LIMITE_DESPUES = r"($|(?=[\s.,;:()\-]))"

def limpiar_comillas_decorativas(s):
    # Quita una comilla/apóstrofe SOLO si está en el borde de una palabra
    # (precedida o seguida de espacio, inicio/fin de cadena, o puntuación
    # como . , ; : ( ) -) -- eso es una comilla decorativa envolviendo un
    # alias (ej. 'PIAGET'. o ALFARO'(CEIGA), donde la comilla de cierre
    # queda pegada a un signo de puntuación en vez de un espacio). Si está
    # pegada entre dos LETRAS (sin espacio ni puntuación de por medio), se
    # preserva: es un apóstrofe ortográfico legítimo de un nombre de origen
    # k'iche' (ej. K'AMOLB'E, CHAAB'IL B'E) o de otro idioma (ej. O'MEANY).
    s = re.sub(LIMITE_ANTES + r"['\"]", "", s)
    s = re.sub(r"['\"]" + LIMITE_DESPUES, "", s)
    return s

antes_comillas = (
    df["ESTABLECIMIENTO"].str.contains('"', regex=False, na=False)
    | df["ESTABLECIMIENTO"].str.contains("'", regex=False, na=False)
).sum()

establecimiento_antes = df["ESTABLECIMIENTO"].copy()
patron_residual = LIMITE_ANTES + r"['\"]|['\"]" + LIMITE_DESPUES

# La regla por posicion (borde de palabra vs. interior) resuelve la enorme
# mayoria de casos con una sola comilla/apostrofe, pero falla en un puñado
# de nombres con comillas DOBLADAS ('' o "" en el mismo borde, ej.
# ''K'AK' AQ ABAL''): al ser una sola pasada de regex, solo retira una capa
# y puede comerse un apostrofe ortografico legitimo de paso. Se detectan
# estos casos probando la regla sobre el valor ORIGINAL (antes de tocar
# nada) y se resuelven con una corrección manual explícita, en vez de
# aplicarles la regla automática y arriesgar perder información.
candidato_automatico = establecimiento_antes.map(limpiar_comillas_decorativas).map(
    lambda s: re.sub(r"\s+", " ", s).strip() if pd.notna(s) else s
)
es_caso_limite = candidato_automatico.str.contains(patron_residual, regex=True, na=False)
casos_limite_raw = sorted(establecimiento_antes.loc[es_caso_limite].unique())
print(f"Casos límite (valor original) que la regla automática no resuelve en una sola pasada ({len(casos_limite_raw)}):")
for r in casos_limite_raw:
    print(" -", repr(r), "-> automático:", repr(limpiar_comillas_decorativas(r)))

CORRECCIONES_MANUALES_ESTABLECIMIENTO = {
    # Comillas dobles envolviendo el nombre completo; el apostrofe final de
    # "YA'" SI es ortografico (glotal, k'iche') y se conserva -- la regla
    # automatica ya lo dejaba bien, se confirma a mano y no se toca mas.
    "CENTRO EDUCATIVO \"RUNAWAL B'ALAM YA'\"": "CENTRO EDUCATIVO RUNAWAL B'ALAM YA'",
    # Comillas simples DOBLADAS ('' ... '') envolviendo el nombre completo;
    # los apostrofes internos de "K'AK'" si son ortograficos (glotales) y se
    # conservan tal como estan en el original, solo se retira la comilla
    # doblada de cada borde (la regla automatica, en una sola pasada, se
    # comia el segundo apostrofe de "K'AK'" junto con la comilla decorativa).
    "INSTITUTO DIVERSIFICADO NORMAL MIXTO POR COOPERATIVA ''K'AK' AQ ÁBÁL''": "INSTITUTO DIVERSIFICADO NORMAL MIXTO POR COOPERATIVA K'AK' AQ ÁBÁL",
    # -- Casos NO detectables por la regla de "borde de palabra" --
    # Una comilla decorativa pegada entre dos letras SIN espacio a ningun
    # lado (ej. "...AGOSTO"FE...") es, por posicion, indistinguible de un
    # apostrofe ortografico legitimo (ej. K'AMOLB'E): ninguna regla
    # posicional puede diferenciarlos, asi que el detector automatico de
    # arriba NO los encuentra. Estos 5 casos se hallaron con una revision
    # manual completa del listado final de nombres con comillas/apostrofe
    # (no por el detector), y se corrigen aqui insertando el espacio que
    # la comilla decorativa se "comio" entre dos palabras.
    "CENTRO EDUCATIVO PRIVADO \"29 DE AGOSTO\"FE Y ALEGRIA NO. 39": "CENTRO EDUCATIVO PRIVADO 29 DE AGOSTO FE Y ALEGRIA NO. 39",
    "COLEGIO EVANGELICO CRISTIANO'RIOS DE AGUA VIVA'": "COLEGIO EVANGELICO CRISTIANO RIOS DE AGUA VIVA",
    "COLEGIO PARTICULAR MIXTO'SACATEPEC'": "COLEGIO PARTICULAR MIXTO SACATEPEC",
    "COLEGIO PRIVADO MIXTO\"SAN JUAN\"": "COLEGIO PRIVADO MIXTO SAN JUAN",
    "\"LICEO CIENCIA Y DESARROLLO\"CON ORIENTACIÓN TÉCNICA Y CIENTÍFICA": "LICEO CIENCIA Y DESARROLLO CON ORIENTACIÓN TÉCNICA Y CIENTÍFICA",
}
assert set(casos_limite_raw) <= set(CORRECCIONES_MANUALES_ESTABLECIMIENTO), (
    "Aparecieron casos límite nuevos que no están cubiertos por las correcciones manuales revisadas: "
    f"{set(casos_limite_raw) - set(CORRECCIONES_MANUALES_ESTABLECIMIENTO)}"
)

es_caso_manual = df["ESTABLECIMIENTO"].isin(CORRECCIONES_MANUALES_ESTABLECIMIENTO)
df.loc[es_caso_manual, "ESTABLECIMIENTO"] = df.loc[es_caso_manual, "ESTABLECIMIENTO"].map(CORRECCIONES_MANUALES_ESTABLECIMIENTO)
df.loc[~es_caso_manual, "ESTABLECIMIENTO"] = df.loc[~es_caso_manual, "ESTABLECIMIENTO"].map(limpiar_comillas_decorativas)
df["ESTABLECIMIENTO"] = normalizar_espacios(df["ESTABLECIMIENTO"])

cambiaron = (establecimiento_antes != df["ESTABLECIMIENTO"]).sum()

# Verificacion final: un solo apostrofe en el borde de una palabra puede ser
# legitimo (glotal, ej. "YA'" al final de una cadena) -- por eso NO se
# vuelve a exigir "cero comillas en borde" aqui (esa regla ya cumplio su
# proposito arriba, para *detectar* candidatos a revisar). Lo que si nunca
# deberia sobrevivir es una comilla/apostrofe DOBLADA ('' o ""): eso es
# inequivocamente un resto de comilla decorativa sin resolver.
patron_doblado = r"['\"]{2,}"
sobrevive_doblado = df["ESTABLECIMIENTO"].str.contains(patron_doblado, regex=True, na=False)
assert sobrevive_doblado.sum() == 0, f"Quedaron {sobrevive_doblado.sum()} comillas dobladas sin resolver"

log.registrar(
    "ESTABLECIMIENTO", f"{antes_comillas} filas con comillas dobles o apóstrofe (mezcla comillas decorativas + apóstrofes legítimos k'iche')",
    "Regla automática (borde de palabra/puntuación vs. interior) + 7 correcciones manuales documentadas: 2 con comillas dobladas que la regla automática no resolvía en una sola pasada, y 5 con una comilla decorativa pegada entre dos letras (indistinguible por posición de un apóstrofe ortográfico), halladas por revisión manual del listado completo",
    cambiaron,
    "La regla automática cubre la enorme mayoría de casos de forma verificable; los 7 nombres residuales se revisaron a mano porque su propio texto ya evidencia cuál es la comilla decorativa y cuál el apóstrofe ortográfico — no se adivina nada, se documenta la decisión de cada uno.",
)
print(f"ESTABLECIMIENTO: {cambiaron} filas modificadas por la regla de comillas (incl. 7 correcciones manuales documentadas).")
df.loc[establecimiento_antes != df['ESTABLECIMIENTO'], ['ESTABLECIMIENTO']].head(5).assign(ANTES=establecimiento_antes[establecimiento_antes != df['ESTABLECIMIENTO']])


Casos límite (valor original) que la regla automática no resuelve en una sola pasada (2):
 - 'CENTRO EDUCATIVO "RUNAWAL B\'ALAM YA\'"' -> automático: "CENTRO EDUCATIVO RUNAWAL B'ALAM YA'"
 - "INSTITUTO DIVERSIFICADO NORMAL MIXTO POR COOPERATIVA ''K'AK' AQ ÁBÁL''" -> automático: "INSTITUTO DIVERSIFICADO NORMAL MIXTO POR COOPERATIVA 'K'AK AQ ÁBÁL'"
[Fase 2 - Texto libre] ESTABLECIMIENTO: Regla automática (borde de palabra/puntuación vs. interior) + 7 correcciones manuales documentadas: 2 con comillas dobladas que la regla automática no resolvía en una sola pasada, y 5 con una comilla decorativa pegada entre dos letras (indistinguible por posición de un apóstrofe ortográfico), halladas por revisión manual del listado completo -> 2936 registros afectados
ESTABLECIMIENTO: 2936 filas modificadas por la regla de comillas (incl. 7 correcciones manuales documentadas).


C:\Users\jplop\AppData\Local\Temp\ipykernel_38968\274052018.py:36: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  es_caso_limite = candidato_automatico.str.contains(patron_residual, regex=True, na=False)


,ESTABLECIMIENTO,ANTES
1,INSTITUTO DE EDUCACION DIVERSIFICADA CENTRO DE ESTUDIOS MERCADOLOGICOS Y PUBLICITARIOS,INSTITUTO DE EDUCACION DIVERSIFICADA 'CENTRO DE ESTUDIOS MERCADOLOGICOS Y PUBLICITARIOS'
3,INSTITUTO DE EDUCACION DIVERSIFICADA LICEO MARIANO GALVEZ,INSTITUTO DE EDUCACION DIVERSIFICADA 'LICEO MARIANO GALVEZ'
14,INSTITUTO NORMAL CASA CENTRAL,INSTITUTO NORMAL 'CASA CENTRAL'
15,INSTITUTO PRIVADO PARA SEÑORITAS DE EDUCACION DIVERSIFICADA SPEEDWRITING (ESCRITURA VE...,INSTITUTO PRIVADO PARA SEÑORITAS DE EDUCACION DIVERSIFICADA 'SPEEDWRITING' (ESCRITURA ...
16,INSTITUTO PRIVADO PARA SEÑORITAS DE EDUCACION DIVERSIFICADA ACADEMIA PRACTICA COMERCIAL,INSTITUTO PRIVADO PARA SEÑORITAS DE EDUCACION DIVERSIFICADA 'ACADEMIA PRACTICA COMERCIAL'


## DIRECCION

In [5]:
antes_vacio_dir = df["DIRECCION"].map(es_vacio).sum()
antes_placeholder_dir = df["DIRECCION"].map(lambda v: es_faltante(v) and not es_vacio(v)).sum()
antes_placeholder_detalle_dir = df.loc[df["DIRECCION"].map(lambda v: es_faltante(v) and not es_vacio(v)), "DIRECCION"].value_counts()
antes_espacio_doble_dir = df["DIRECCION"].str.contains(r"  ", regex=True, na=False).sum()
print(f"Placeholders de faltante encontrados en DIRECCION (además de vacío real):\n{antes_placeholder_detalle_dir}")
antes_minuscula_dir = df["DIRECCION"].str.contains(r"[a-z]", regex=True, na=False).sum()

df["DIRECCION"] = normalizar_espacios(df["DIRECCION"])
df["DIRECCION"] = df["DIRECCION"].where(df["DIRECCION"].isna() | (df["DIRECCION"] == ""), df["DIRECCION"].str.upper())

assert not df["DIRECCION"].dropna().str.contains(r"  ", regex=True).any(), "Quedaron espacios dobles en DIRECCION"

log.registrar(
    "DIRECCION", f"{antes_espacio_doble_dir} filas con espacios dobles; {antes_minuscula_dir} filas con minúsculas sueltas",
    "Trim + colapso de espacios; mayúsculas consistentes en todo el campo", antes_espacio_doble_dir + antes_minuscula_dir,
    "Normaliza formato sin alterar el contenido semántico de la dirección.",
)
print(f"DIRECCION: {antes_espacio_doble_dir} espacios dobles y {antes_minuscula_dir} filas en minúscula corregidas.")


Placeholders de faltante encontrados en DIRECCION (además de vacío real):
DIRECCION
.     5
--    1
Name: count, dtype: int64


[Fase 2 - Texto libre] DIRECCION: Trim + colapso de espacios; mayúsculas consistentes en todo el campo -> 495 registros afectados
DIRECCION: 485 espacios dobles y 10 filas en minúscula corregidas.


In [6]:
def estandarizar_abreviaturas_direccion(s):
    if pd.isna(s):
        return s
    s = re.sub(r"\bZ\.\s*", "ZONA ", s)
    s = re.sub(r"\bAV\.\s*", "AVENIDA ", s)
    s = re.sub(r"\bCALZ\.\s*", "CALZADA ", s)
    s = re.sub(r"\bKILOMETRO\.?\s*", "KM. ", s)
    s = re.sub(r"\bKM\b(?!\.)\s*", "KM. ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

direccion_antes = df["DIRECCION"].copy()
df["DIRECCION"] = df["DIRECCION"].map(estandarizar_abreviaturas_direccion)
cambiaron_abrev = (direccion_antes != df["DIRECCION"]).sum()

for patron, etiqueta in [(r"\bZ\.", "Z."), (r"\bAV\.", "AV."), (r"\bCALZ\.", "CALZ."), (r"\bKILOMETRO", "KILOMETRO"), (r"\bKM\b(?!\.)", "KM sin punto")]:
    restante = df["DIRECCION"].dropna().str.contains(patron, regex=True).sum()
    assert restante == 0, f"Quedaron {restante} casos de la abreviatura '{etiqueta}' sin estandarizar"

log.registrar(
    "DIRECCION", "Abreviaturas inconsistentes: Z./ZONA, AV./AVENIDA, CALZ./CALZADA, KM/KM./KILOMETRO",
    "Se estandariza cada grupo a la forma ya dominante en el propio dataset: Z.->ZONA, AV.->AVENIDA, CALZ.->CALZADA, KILOMETRO/KM->'KM.'",
    cambiaron_abrev,
    "Se preserva la redacción original en todo lo demás (no se fuerza un formato postal estricto); solo se unifican estas 4 abreviaturas de alta frecuencia, hacia la forma que ya predomina en los datos para minimizar el cambio total.",
)
print(f"DIRECCION: {cambiaron_abrev} filas con abreviaturas estandarizadas.")


[Fase 2 - Texto libre] DIRECCION: Se estandariza cada grupo a la forma ya dominante en el propio dataset: Z.->ZONA, AV.->AVENIDA, CALZ.->CALZADA, KILOMETRO/KM->'KM.' -> 565 registros afectados
DIRECCION: 565 filas con abreviaturas estandarizadas.


In [7]:
df["DIRECCION"] = marcar_faltantes_como_na(df["DIRECCION"])
despues_na_dir = df["DIRECCION"].isna().sum()

assert despues_na_dir == antes_vacio_dir + antes_placeholder_dir, (
    f"NA de DIRECCION ({despues_na_dir}) no coincide con vacíos+placeholder originales ({antes_vacio_dir + antes_placeholder_dir})"
)

log.registrar(
    "DIRECCION", f"{antes_vacio_dir} celdas vacías + {antes_placeholder_dir} con placeholders de faltante (\"--\", \".\", etc.)",
    "Se marcan como NA explícito (vacío real, no se imputa ninguna dirección)", antes_vacio_dir + antes_placeholder_dir,
    "Ninguno de estos placeholders es una dirección válida bajo ninguna lectura razonable (ver guía, Actividad 5.a.iv); se tratan igual que una celda vacía.",
)
print(f"DIRECCION: {despues_na_dir} valores marcados NA (vacíos + placeholders).")


[Fase 2 - Texto libre] DIRECCION: Se marcan como NA explícito (vacío real, no se imputa ninguna dirección) -> 82 registros afectados
DIRECCION: 82 valores marcados NA (vacíos + placeholders).


## SUPERVISOR

In [8]:
antes_vacio_sup = df["SUPERVISOR"].map(es_vacio).sum()
antes_espacio_doble_sup = df["SUPERVISOR"].str.contains(r"  ", regex=True, na=False).sum()

df["SUPERVISOR"] = normalizar_espacios(df["SUPERVISOR"])
df["SUPERVISOR"] = marcar_faltantes_como_na(df["SUPERVISOR"], tratar_placeholders=False)

despues_na_sup = df["SUPERVISOR"].isna().sum()
assert despues_na_sup == antes_vacio_sup, f"NA de SUPERVISOR ({despues_na_sup}) no coincide con vacíos originales ({antes_vacio_sup})"
assert not df["SUPERVISOR"].dropna().str.contains(r"  ", regex=True).any(), "Quedaron espacios dobles en SUPERVISOR"

log.registrar(
    "SUPERVISOR", f"{antes_vacio_sup} celdas vacías; {antes_espacio_doble_sup} con espacios dobles",
    "Trim + colapso de espacios; vacíos a NA explícito. No se imputa ningún nombre", antes_vacio_sup + antes_espacio_doble_sup,
    "Mismo tratamiento que el resto de campos de nombre propio: no se inventa información faltante.",
)
print(f"SUPERVISOR: {despues_na_sup} NA, {antes_espacio_doble_sup} espacios dobles corregidos.")


[Fase 2 - Texto libre] SUPERVISOR: Trim + colapso de espacios; vacíos a NA explícito. No se imputa ningún nombre -> 633 registros afectados
SUPERVISOR: 535 NA, 98 espacios dobles corregidos.


## DIRECTOR

In [9]:
antes_vacio_dir2 = df["DIRECTOR"].map(es_vacio).sum()
antes_placeholder_dir2 = df["DIRECTOR"].map(lambda v: es_faltante(v) and not es_vacio(v)).sum()
antes_placeholder_detalle_dir2 = df.loc[df["DIRECTOR"].map(lambda v: es_faltante(v) and not es_vacio(v)), "DIRECTOR"].value_counts()
antes_espacio_doble_dir2 = df["DIRECTOR"].str.contains(r"  ", regex=True, na=False).sum()
print(f"Placeholders de faltante encontrados en DIRECTOR (además de vacío real):\n{antes_placeholder_detalle_dir2}")

df["DIRECTOR"] = normalizar_espacios(df["DIRECTOR"])
df["DIRECTOR"] = marcar_faltantes_como_na(df["DIRECTOR"])

despues_na_dir2 = df["DIRECTOR"].isna().sum()
assert despues_na_dir2 == antes_vacio_dir2 + antes_placeholder_dir2, (
    f"NA de DIRECTOR ({despues_na_dir2}) no coincide con vacíos+placeholder originales ({antes_vacio_dir2 + antes_placeholder_dir2})"
)
assert not df["DIRECTOR"].dropna().str.contains(r"  ", regex=True).any(), "Quedaron espacios dobles en DIRECTOR"

log.registrar(
    "DIRECTOR", f"{antes_vacio_dir2} celdas vacías (14.6%, la variable con más faltantes) + {antes_placeholder_dir2} con placeholders de faltante (\"--\", \"SIN DATO\", \"-\", \".\"); {antes_espacio_doble_dir2} con espacios dobles",
    "Trim + colapso de espacios; vacíos y placeholders a NA explícito. No se imputa ningún nombre", antes_vacio_dir2 + antes_placeholder_dir2 + antes_espacio_doble_dir2,
    "Ninguno de estos placeholders es un nombre válido (ver guía, Actividad 5.a.iv); se tratan igual que una celda vacía. No se imputa bajo ninguna circunstancia (generaría datos falsos).",
)
print(f"DIRECTOR: {despues_na_dir2} NA (vacíos + placeholders), {antes_espacio_doble_dir2} espacios dobles corregidos.")


Placeholders de faltante encontrados en DIRECTOR (además de vacío real):
DIRECTOR
--          41
SIN DATO    28
-           16
.           14
Name: count, dtype: int64
[Fase 2 - Texto libre] DIRECTOR: Trim + colapso de espacios; vacíos y placeholders a NA explícito. No se imputa ningún nombre -> 2697 registros afectados
DIRECTOR: 1831 NA (vacíos + placeholders), 866 espacios dobles corregidos.


## Guardar salida de la Fase 2

In [10]:
ruta_salida = os.path.join(CARPETA_INTERIM, "fase2_texto_libre.csv")
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

ruta_log = os.path.join(CARPETA_INTERIM, "log_fase2.csv")
log.guardar(ruta_log)

print(f"\nFase 2 completa: {df.shape[0]} filas, {df.shape[1]} columnas -> {ruta_salida}")
log.a_dataframe()


Registro de transformaciones de Fase 2 - Texto libre guardado en: C:\Users\jplop\Downloads\DATAS\scripts\..\data\interim\log_fase2.csv

Fase 2 completa: 11867 filas, 18 columnas -> C:\Users\jplop\Downloads\DATAS\scripts\..\data\interim\fase2_texto_libre.csv


,fase,variable,problema_detectado,transformacion,registros_afectados,justificacion
0,Fase 2 - Texto libre,ESTABLECIMIENTO,1392 filas con espacios dobles internos,Trim + colapso de espacios múltiples a uno solo,1392,"Limpieza de formato estándar; no cambia el contenido, solo el espaciado."
1,Fase 2 - Texto libre,ESTABLECIMIENTO,Mayúsculas: se verifica consistencia (no se fuerza un cambio de estilo innecesario),Sin cambios: el campo ya está 100% en mayúsculas,0,La fuente ya entrega el campo casi enteramente en mayúsculas; forzar mayúsculas sobre ...
2,Fase 2 - Texto libre,ESTABLECIMIENTO,2959 filas con comillas dobles o apóstrofe (mezcla comillas decorativas + apóstrofes l...,Regla automática (borde de palabra/puntuación vs. interior) + 7 correcciones manuales ...,2936,La regla automática cubre la enorme mayoría de casos de forma verificable; los 7 nombr...
3,Fase 2 - Texto libre,DIRECCION,485 filas con espacios dobles; 10 filas con minúsculas sueltas,Trim + colapso de espacios; mayúsculas consistentes en todo el campo,495,Normaliza formato sin alterar el contenido semántico de la dirección.
4,Fase 2 - Texto libre,DIRECCION,"Abreviaturas inconsistentes: Z./ZONA, AV./AVENIDA, CALZ./CALZADA, KM/KM./KILOMETRO","Se estandariza cada grupo a la forma ya dominante en el propio dataset: Z.->ZONA, AV.-...",565,Se preserva la redacción original en todo lo demás (no se fuerza un formato postal est...
5,Fase 2 - Texto libre,DIRECCION,"76 celdas vacías + 6 con placeholders de faltante (""--"", ""."", etc.)","Se marcan como NA explícito (vacío real, no se imputa ninguna dirección)",82,Ninguno de estos placeholders es una dirección válida bajo ninguna lectura razonable (...
6,Fase 2 - Texto libre,SUPERVISOR,535 celdas vacías; 98 con espacios dobles,Trim + colapso de espacios; vacíos a NA explícito. No se imputa ningún nombre,633,Mismo tratamiento que el resto de campos de nombre propio: no se inventa información f...
7,Fase 2 - Texto libre,DIRECTOR,"1732 celdas vacías (14.6%, la variable con más faltantes) + 99 con placeholders de fal...",Trim + colapso de espacios; vacíos y placeholders a NA explícito. No se imputa ningún ...,2697,"Ninguno de estos placeholders es un nombre válido (ver guía, Actividad 5.a.iv); se tra..."
